## Data loading and cleaning

In [ ]:
import hvplot.pandas # noqa
import json
import numpy as np
import pandas as pd
import panel as pn
pd.set_option('future.no_silent_downcasting', True)

In [ ]:
with open('yearly_metrics.json') as f:
    data = json.load(f)


df = pd.DataFrame(data["issues"])
df['created_at'] = pd.to_datetime(df['created_at']).dt.tz_convert(None)
df.set_index('created_at', inplace=True)
df.replace("None", np.nan, inplace=True)
df.head(3)

## Analysis

In [ ]:
total = data["total_item_count"]
opened = data["num_items_opened"]
closed = data["num_items_closed"]
first_month = df.index[-1].strftime('%B %Y')
last_month = df.index[0].strftime('%B %Y')

In [ ]:
import re
import datetime

def parse_timedelta(time_str):
    """Convert time strings to timedelta"""
    # This regex looks for an optional "X day(s), " followed by "HH:MM:SS"
    pattern = r'(?:(\d+)\s+days?,\s+)?(\d+):(\d+):(\d+)'
    match = re.match(pattern, time_str)
    if match:
        days = int(match.group(1)) if match.group(1) else 0
        hours = int(match.group(2))
        minutes = int(match.group(3))
        seconds = int(match.group(4))
        return datetime.timedelta(days=days, hours=hours, minutes=minutes, seconds=seconds)
    else:
        raise ValueError("The time string format is not recognized.")

close_time = parse_timedelta(data["average_time_to_close"])
median_close_time = parse_timedelta(data["median_time_to_close"])

## Visualization

In [ ]:
# Plots

monthly_opened = df.resample('ME').size()
monthly_closed =  df.dropna(subset=['time_to_close']).resample('ME').size()
comparison_df = pd.DataFrame({
    'Opened': monthly_opened,
    'Closed': monthly_closed
})
comparison_plot = comparison_df.hvplot.line(xlabel="Month", ylabel="Number of Issues",
                                            title="Opened vs Closed Issues per Month")
cumulative_open_issues = df.dropna(subset=['time_to_close']).resample('D').size().cumsum()
cumulative_issues_plot = cumulative_open_issues.hvplot.line(xlabel="Date", ylabel="Cumulative Issues",
                                                            title="Cumulative Open Issues Over Time")

author_counts = df.groupby('author').size().sort_values(ascending=False)[:10]
author_plot = author_counts.hvplot.bar(
    xlabel="Author", ylabel="Number of Issues", title="Authors with the most Issues", rot=45,
)

In [ ]:
# Dashboard

pn.extension('tabulator')

styles = {
    "box-shadow": "rgba(50, 50, 93, 0.25) 0px 6px 12px -2px, rgba(0, 0, 0, 0.3) 0px 3px 7px -3px",
    "border-radius": "5px",
    "padding": "10px",
}

indicators = pn.FlexBox(
    pn.indicators.Number(
        value=total,
        name="Total Issues",
        default_color="blue",
        styles=styles,
    ),
    pn.indicators.Number(
        value=opened,
        name="Issues opened",
        default_color="green",
        styles=styles
    ),
    pn.indicators.Number(
        value=closed,
        name="Issues closed",
        default_color="red",
        styles=styles,
    ),
    pn.indicators.Number(
        value=close_time.days,
        name="Avg. time to close (days)",
        default_color="gray",
        styles=styles,
    ),
    pn.indicators.Number(
        value=median_close_time.days,
        name="Median time to close (days)",
        default_color="blue",
        styles=styles,
    ),
)
# Shorten 'html_url' column and convert to clickable links
df['issue_no'] = df['html_url'].apply(
    lambda url: f'<a href="{url}" target="_blank">{url.split("/")[-1]}</a>'
)

issue_filter = pn.widgets.TextInput(name="Filter by Issue Number", placeholder="6552")
status_filter = pn.widgets.Select(name="Filter by Issue Status", options=["All", "Open", "Closed"], value="All")

# Function to return the filtered DataFrame based on widget values:
def get_filtered_df():
    filtered = df.copy()
    if issue_filter.value:
        # Filter the 'issue_no' column (converted to string for partial matching)
        filtered = filtered[filtered['issue_no'].astype(str).str.contains(issue_filter.value)]
    if status_filter.value != "All":
        if status_filter.value == "Open":
            # Use pd.isna() to identify open issues (i.e., missing time_to_close)
            filtered = filtered[filtered['time_to_close'].isna()]
        elif status_filter.value == "Closed":
            # Use notna() to identify closed issues (i.e., having a time_to_close value)
            filtered = filtered[filtered['time_to_close'].notna()]
    return filtered

table = pn.widgets.Tabulator(
    df,
    sizing_mode="stretch_width",
    name="Table",
    hidden_columns=['label_metrics', 'html_url', 'time_to_answer', 'time_in_draft'],
    pagination='remote',
    page_size=5,
    formatters={'issue_no': 'html'}
)

# Callback function to update the table based on filter changes:
def update_table(event):
    table.value = get_filtered_df()

# Watch for changes in the filter widgets and update the table:
issue_filter.param.watch(update_table, 'value')
status_filter.param.watch(update_table, 'value')

tabs = pn.Tabs(
    ('Open vs Closed Issues', comparison_plot),
    ('Issue Authors', author_plot),
    ('Open Issues over time', cumulative_issues_plot),
    sizing_mode="scale_both", margin=10
)

logo = '<img src="https://holoviz.org/_static/holoviz-logo.svg">'


text = f"""
[Panel](https://panel.holoviz.org) dashboard with Issue metrics from {first_month} to {last_month}.
"""

template = pn.template.FastListTemplate(
    title="Holoviews Issue Metrics dashboard",
    sidebar=[logo, text, issue_filter, status_filter],
    collapsed_sidebar = True,
    main=[pn.Column('# Summary Insights ', indicators, '# Data table', table, '# Plots', tabs)],
    main_layout=None,
    accent="#4099da",
)

template.servable();